# Psychological Risk Assessment Framework for University Students

This interactive notebook demonstrates the complete machine learning workflow for university student psychological risk classification.

> **Dataset Disclosure**: This framework utilizes a synthetic dataset ($N=400$ student cohorts) generated via probabilistic simulation for research and pipeline testing purposes.
>
> **Non-Clinical Disclaimer**: This project represents an academic early-warning risk classification framework. It is **not** a clinical diagnostic system and should not be used to diagnose mental health conditions.

## 1. Imports & Environment Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local src modules are resolvable
sys.path.append('..') if 'src' not in os.listdir('.') else None

from src.data_preprocessing import load_dataset, generate_risk_labels, prepare_features_and_target
from src.models import get_verified_model_pipelines, run_cross_validation, evaluate_models_on_test_set
from src.recommendation import format_student_assessment_report

print("Environment initialized successfully.")

## 2. Data Loading & 5-Class Target Risk Label Generation

Target categories are derived programmatically using a distress-protective balance equation:

$$\text{Distress} = \frac{\text{well\_depressive\_mood}}{3.0} + \frac{\text{well\_anxious\_mood}}{3.0} + \dots$$
$$\text{Protective} = \frac{\text{well\_optimism}}{3.0} + \dots$$
$$r_{\text{total}} = \text{Distress} - \text{Protective}$$

- **Low Risk (0)**: $r_{\text{total}} \le -3.0$
- **Mild Risk (1)**: $-3.0 < r_{\text{total}} \le -2.0$
- **Moderate Risk (2)**: $-2.0 < r_{\text{total}} \le -1.0$
- **High Risk (3)**: $-1.0 < r_{\text{total}} \le 0.0$
- **Critical Risk (4)**: $r_{\text{total}} > 0.0$

In [ ]:
df = load_dataset("../data/raw/student_mental_health.csv" if os.path.exists("../data/raw/student_mental_health.csv") else "data/raw/student_mental_health.csv")
df = generate_risk_labels(df)

class_names = ['Low Risk', 'Mild Risk', 'Moderate Risk', 'High Risk', 'Critical Risk']
class_counts = df['Risk_Class'].value_counts().sort_index()

print(f"Loaded dataset with {len(df)} student cohorts.")
for idx, count in class_counts.items():
    print(f"  - Class {idx} ({class_names[idx]}): {count} ({count/len(df)*100:.2f}%)")

## 3. Preprocessing, Feature Engineering & Target Leakage Prevention

To prevent severe target leakage, the 4 wellbeing subscale features (`well_depressive_mood`, `well_anxious_mood`, `well_emotional_coping`, `well_overwhelmed`) are explicitly dropped from predictor variables before model fitting.

Five domain-specific features are engineered:
1. **Academic Stress Index**
2. **Lifestyle Score**
3. **Social Support Score**
4. **Sleep Quality Indicator**
5. **Behavioral Consistency Score**

In [ ]:
X_train, X_test, y_train, y_test = prepare_features_and_target(df, test_size=0.2, random_state=42)

categorical_features = ['Gender', 'Living_Status', 'fin_work_status', 'Coping_Mechanism', 'env_counseling_use']
categorical_features = [col for col in categorical_features if col in X_train.columns]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

num_cols_extended = numeric_features + [
    'Academic_Stress_Index', 'Lifestyle_Score', 
    'Social_Support_Score', 'Sleep_Quality_Indicator', 
    'Behavioral_Consistency_Score'
]

print(f"Training set shape: {X_train.shape}")
print(f"Holdout test set shape: {X_test.shape}")

## 4. Holdout Test Set Evaluation & Results Table

In [ ]:
pipelines = get_verified_model_pipelines(numeric_features, categorical_features, num_cols_extended, random_state=42)

test_metrics, feature_importances, proposed_estimator = evaluate_models_on_test_set(
    pipelines, X_train, y_train, X_test, y_test,
    numeric_features, categorical_features, num_cols_extended
)

res_rows = []
for m_name in ['Random Forest', 'XGBoost', 'CatBoost', 'Proposed Framework']:
    res_rows.append({
        "Model": m_name,
        "Accuracy": f"{test_metrics[m_name]['Accuracy']*100:.2f}%",
        "Precision (Macro)": f"{test_metrics[m_name]['Precision']*100:.2f}%",
        "Recall (Macro)": f"{test_metrics[m_name]['Recall']*100:.2f}%",
        "F1-Score (Macro)": f"{test_metrics[m_name]['F1-Score']*100:.2f}%",
        "ROC-AUC (Macro)": f"{test_metrics[m_name]['ROC-AUC']*100:.2f}%"
    })

df_results = pd.DataFrame(res_rows)
display(df_results)

## 5. Visualizations & Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
models_list = ['Random Forest', 'XGBoost', 'CatBoost', 'Proposed Framework']
accuracies = [test_metrics[m]['Accuracy'] * 100 for m in models_list]
colors = ['#b0c4de', '#8ba8c7', '#5f84a2', '#1f4e79']

bars = ax.bar(models_list, accuracies, color=colors, edgecolor='black', width=0.45)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Comparison of Accuracy Across Models', fontweight='bold', pad=10)
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Early Warning Recommendation Demonstration

In [ ]:
sample_probs = proposed_estimator.predict_proba(X_test.iloc[0:1])[0]
sample_pred = np.argmax(sample_probs)
sample_report = format_student_assessment_report("STU-2026-0042", sample_pred, sample_probs)
print(sample_report)